In [1]:
from __future__ import annotations

import jax.numpy as jnp
import numpy as np
import dynamiqs as dq

from parameters import DeviceParameters, PulseParameters
from hamiltonian import operators

In [2]:
from parameters import load_json

# Load the JSON
device, pulse, simulation = load_json("device_parameter.json")

# -------------------------------
# Print the loaded objects
# -------------------------------

print("===== Device =====")
print(device)
print()

print("===== Pulse =====")
print(pulse)
print()

print("===== Simulation =====")
print(simulation)
print()

# -------------------------------
# Print individual pulse fields
# -------------------------------

print("omega_d   :", pulse.omega_d)
print("epsilon1  :", pulse.epsilon1)
print("epsilon2  :", pulse.epsilon2)
print("t_switch  :", pulse.t_switch)
print("t_off     :", pulse.t_off)
print()

# -------------------------------
# Test the modulation manually
# -------------------------------

import jax.numpy as jnp

def modulation(t):
    amp = jnp.where(
        t < pulse.t_switch,
        pulse.epsilon1,
        jnp.where(
            t < pulse.t_off,
            pulse.epsilon2,
            0.0,
        ),
    )
    return 0.5 * amp

print("===== Modulation values =====")

for t in [0, 50, 99, 100, 150, 299, 300, 301, 500]:
    print(f"t = {t:3d} ns   ->   {float(modulation(t))}")

===== Device =====
DeviceParameters(omega_levels=array([  0.        ,   2.44125645,  24.51829029,  38.39085932,
        57.69726277,  76.83614993,  96.00748727, 113.94813516,
       129.66739968, 142.67212914, 154.27552557, 166.48430725,
       179.83159958, 193.90649291, 208.51475336, 223.61841799,
       239.15804494, 255.06381514, 271.2739293 , 287.72823444,
       304.36415457, 321.11614088]), n_matrix_real=array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 

In [9]:
print(pulse.epsilon1/(2*np.pi))
print(pulse.epsilon2/(2*np.pi))

0.03
0.016249999999999997


In [10]:
print(device.g/(2*np.pi))

0.2889999999999999


In [11]:
import numpy as np

epsilon = 0.251327
kappa = 0.031415926535897934

print((epsilon / kappa) ** 2)

63.99979002403392


In [12]:
ops = operators(device, pulse)

test = ops["drive_op"] - (-1j)*(ops["a"] - ops["a"].dag())

print(np.max(np.abs(np.asarray(test))))

/mnt/d/GPU_fluxonium/hamiltonian.py:55: UserWarning: A sparse qarray has been converted to dense layout due to element-wise addition with a dense qarray.
  "h_static": h_resonator + h_fluxonium + params.g * drive_op @ n,


0.0


In [13]:
import dynamiqs as dq
print(dq.__version__)

0.3.4


In [14]:
ops = operators(device)
print(ops["drive_op"])

TypeError: operators() missing 1 required positional argument: 'pulse'

In [10]:
ops = operators(device)

print("a:")
print(ops["a"])

print("drive_op:")
print(ops["drive_op"])

print("a + adag:")
print(ops["a"] + ops["a"].dag())

TypeError: operators() missing 1 required positional argument: 'pulse'

In [ ]:
import numpy as np

A = np.array(ops["drive_op"])

print(np.max(np.abs(np.real(A))))
print(np.max(np.abs(np.imag(A))))

NameError: name 'ops' is not defined

In [15]:
print(ops["a"])
print(ops["drive_op"])
print(-1j * (ops["a"] - ops["a"].dag()))
print(ops["a"] + ops["a"].dag())

QArray: shape=(1430, 1430), dims=(22, 65), dtype=complex64, layout=dia, ndiags=1
[[      ⋅       1.       +0.j       ⋅       ...       ⋅      
        ⋅             ⋅      ]
 [      ⋅             ⋅       1.4142135+0.j ...       ⋅      
        ⋅             ⋅      ]
 [      ⋅             ⋅             ⋅       ...       ⋅      
        ⋅             ⋅      ]
 ...
 [      ⋅             ⋅             ⋅       ...       ⋅      
  7.937254 +0.j       ⋅      ]
 [      ⋅             ⋅             ⋅       ...       ⋅      
        ⋅       8.       +0.j]
 [      ⋅             ⋅             ⋅       ...       ⋅      
        ⋅             ⋅      ]]
QArray: shape=(1430, 1430), dims=(22, 65), dtype=complex64, layout=dia, ndiags=2
[[  ⋅           0.-1.j          ⋅           ...   ⋅   
    ⋅             ⋅          ]
 [0.+1.j          ⋅           0.-1.4142135j ...   ⋅   
    ⋅             ⋅          ]
 [  ⋅           0.+1.4142135j   ⋅           ...   ⋅   
    ⋅             ⋅          ]
 ...
 [  ⋅      

In [16]:
import inspect
import dynamiqs as dq

print(inspect.signature(dq.mesolve))

(H: 'QArrayLike | TimeQArray', jump_ops: 'list[QArrayLike | TimeQArray]', rho0: 'QArrayLike', tsave: 'ArrayLike', *, exp_ops: 'list[QArrayLike] | None' = None, method: 'Method' = Tsit5(), gradient: 'Gradient | None' = None, options: 'Options' = Options()) -> 'MESolveResult'


In [17]:
import inspect
import dynamiqs as dq

print(inspect.signature(dq.Tsit5))

AttributeError: module 'dynamiqs' has no attribute 'Tsit5'

In [18]:
import dynamiqs as dq

print(dir(dq))

['DSMESolveResult', 'DSSESolveResult', 'DenseQArray', 'FloquetResult', 'JSMESolveResult', 'JSSESolveResult', 'MEPropagatorResult', 'MESolveResult', 'Options', 'QArray', 'QArrayLike', 'SEPropagatorResult', 'SESolveResult', 'SparseDIAQArray', 'TimeQArray', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_checks', '_utils', 'asqarray', 'basis', 'basis_dm', 'bloch_coordinates', 'braket', 'cd_gate', 'clicktimes_sse_to_sme', 'cnot', 'coherent', 'coherent_dm', 'constant', 'cosm', 'create', 'dag', 'dense', 'dense_qarray', 'destroy', 'dia', 'displace', 'dissipator', 'dsmesolve', 'dssesolve', 'entropy_relative', 'entropy_vn', 'excited', 'excited_dm', 'expect', 'expm', 'eye', 'eye_like', 'fidelity', 'floquet', 'fock', 'fock_dm', 'general', 'global_settings', 'gradient', 'ground', 'ground_dm', 'hadamard', 'hc', 'helpers', 'hermitian_conjugate', 'integrators', 'isbra', 'isdm', 'isherm', 'isket', 'isop', 'isqarrayl

In [19]:
print(type(dq.mesolve))

<class 'function'>


In [20]:
help(dq.mesolve)

Help on function mesolve in module dynamiqs.integrators.apis.mesolve:

mesolve(H: 'QArrayLike | TimeQArray', jump_ops: 'list[QArrayLike | TimeQArray]', rho0: 'QArrayLike', tsave: 'ArrayLike', *, exp_ops: 'list[QArrayLike] | None' = None, method: 'Method' = Tsit5(), gradient: 'Gradient | None' = None, options: 'Options' = Options()) -> 'MESolveResult'
    Solve the Lindblad master equation.

    This function computes the evolution of the density matrix $\rho(t)$ at time $t$,
    starting from an initial state $\rho_0$, according to the Lindblad master
    equation (with $\hbar=1$ and where time is implicit(1))
    $$
        \frac{\dd\rho}{\dt} = -i[H, \rho]
        + \sum_{k=1}^N \left(
            L_k \rho L_k^\dag
            - \frac{1}{2} L_k^\dag L_k \rho
            - \frac{1}{2} \rho L_k^\dag L_k
        \right),
    $$
    where $H$ is the system's Hamiltonian and $\{L_k\}$ is a collection of jump
    operators.
    { .annotate }

    1. With explicit time dependence:
        -

In [21]:
from dynamiqs.method import Tsit5
import inspect

print(inspect.signature(Tsit5))

(rtol: 'float' = 1e-06, atol: 'float' = 1e-06, safety_factor: 'float' = 0.9, min_factor: 'float' = 0.2, max_factor: 'float' = 5.0, max_steps: 'int' = 100000)


In [22]:
from dynamiqs.method import Dopri8
import inspect
print(inspect.signature(Dopri8))

(rtol: 'float' = 1e-06, atol: 'float' = 1e-06, safety_factor: 'float' = 0.9, min_factor: 'float' = 0.2, max_factor: 'float' = 5.0, max_steps: 'int' = 100000)


In [23]:
ops = operators(device)

print(type(ops["h_static"]))
print(ops["h_static"])

TypeError: operators() missing 1 required positional argument: 'pulse'

In [ ]:
ops = operators(device)

print(type(ops["a"]))
print(type(ops["h_static"]))
print(type(ops["n"]))

<class 'dynamiqs.qarrays.sparsedia_qarray.SparseDIAQArray'>
<class 'dynamiqs.qarrays.sparsedia_qarray.SparseDIAQArray'>
<class 'dynamiqs.qarrays.sparsedia_qarray.SparseDIAQArray'>


In [ ]:
from hamiltonian import (readout_hamiltonian,
                         readout_hamiltonian_rwa,
                         basis_state,
                         operators
                         )

def hamiltonian_diagonalize(params=DeviceParameters, pulse = PulseParameters)-> dq.TimeQArray:
    ops = operators(params, pulse)
    h_static = ops["h_static"]
    h_interaction_rwa = params.g * (-1j)*( ops["n_plus"] @ ops["a"] - ops["n_minus"] @ (ops["a"].dag()))
    h_dressed_rwa = h_static + h_interaction_rwa
    h_interaction = params.g * (-1j)*(ops["a"] - ops["a"].dag()) @ ops["n"]
    h_dressed = h_static + h_interaction
    diagonalized = np.diag(h_dressed)
    H = h_dressed.to_jax()
    H = np.asarray(H)

    # Diagonalize
    evals, evecs = np.linalg.eigh(H)
    return evals, evecs
    #print(diagonalized)
    #print(h_dressed_rwa)
    #print(h_static)

In [ ]:
a, b = hamiltonian_diagonalize(device, pulse)

/mnt/d/GPU_fluxonium/hamiltonian.py:40: UserWarning: A sparse qarray has been converted to dense layout due to element-wise addition with a dense qarray.
  "h_static": h_resonator + h_fluxonium + params.g * drive_op @ n,


In [ ]:
g0 = basis_state(device , 0, 0)
g1 = basis_state(device , 0, 1)
e0 = basis_state(device , 1, 0)
e1 = basis_state(device , 1, 1)
overlaps = np.abs(b.conj().T @ g0)**2
idx_g0 = np.argmax(overlaps)
overlaps = np.abs(b.conj().T @ g1)**2
idx_g1 = np.argmax(overlaps)
overlaps = np.abs(b.conj().T @ e0)**2
idx_e0 = np.argmax(overlaps)
overlaps = np.abs(b.conj().T @ e1)**2
idx_e1 = np.argmax(overlaps)
omega_r_g = a[idx_g1] - a[idx_g0]
omega_r_e = a[idx_e1] - a[idx_e0]
print(omega_r_g , omega_r_e)


58.87512 58.8929


In [ ]:
def omega_d(params= DeviceParameters, pulse = PulseParameters) -> int:
    ops = operators(params, pulse)
    h_static = ops["h_static"]
    h_interaction_rwa = params.g * (-1j)*( ops["n_plus"] @ ops["a"] - ops["n_minus"] @ (ops["a"].dag()))
    h_dressed_rwa = h_static + h_interaction_rwa
    h_interaction = params.g * (-1j)*(ops["a"] - ops["a"].dag()) @ ops["n"]
    h_dressed = h_static + h_interaction
    diagonalized = np.diag(h_dressed)
    H = h_dressed.to_jax()
    H = np.asarray(H)    
    evals, evecs = np.linalg.eigh(H)
    g0 = basis_state(device , 0, 0)
    g1 = basis_state(device , 0, 1)
    e0 = basis_state(device , 1, 0)
    e1 = basis_state(device , 1, 1)
    overlaps = np.abs(b.conj().T @ g0)**2
    idx_g0 = np.argmax(overlaps)
    overlaps = np.abs(b.conj().T @ g1)**2
    idx_g1 = np.argmax(overlaps)
    overlaps = np.abs(b.conj().T @ e0)**2
    idx_e0 = np.argmax(overlaps)
    overlaps = np.abs(b.conj().T @ e1)**2
    idx_e1 = np.argmax(overlaps)
    omega_r_g = a[idx_g1] - a[idx_g0]
    omega_r_e = a[idx_e1] - a[idx_e0]
    print(omega_r_g , omega_r_e)
    return omega_r_g, omega_r_e


In [ ]:
a, b = hamiltonian_diagonalize(device, pulse)


/mnt/d/GPU_fluxonium/hamiltonian.py:40: UserWarning: A sparse qarray has been converted to dense layout due to element-wise addition with a dense qarray.
  "h_static": h_resonator + h_fluxonium + params.g * drive_op @ n,


TypeError: tuple indices must be integers or slices, not tuple

In [ ]:
b[0,1]

np.complex64(-2.3331466e-17+0j)

In [ ]:
def dressed_fluxonium(params: DeviceParameters, pulse: PulseParameters) -> dq.QArray:
    h_resonator = params.omega_r*ops["n_photon"]
    h_fluxonium = ops["h_fluxonium"]
    h_interaction_rwa = params.g*(-1j)*(ops["n_plus"] @ ops["a"] - ops["n_minus"] @ ops["a"].dag())
    h_interaction = params.g*(-1j)*(ops["a"] - ops["a"].dag()) @ ops["n"]
    h_dressed = h_resonator + h_fluxonium + h_interaction_rwa
    H = h_dressed.to_jax()
    H = np.asarray(H)
    evals, evecs = np.linalg.eigh(H)
    
    g0 = np.asarray(basis_state(params, 0, 0).to_jax()).flatten()
   

    # overlaps with all dressed states
    overlaps_g0 = np.abs(evecs.conj().T @ g0)**2
    
    idx_g0 = np.argmax(overlaps_g0)
    

    
    P_g0 = np.outer(evecs[:, idx_g0], evecs[:, idx_g0].conj())
    
    P_comp = P_g0 
    for i in range(1, device.resonator_dim):
        n0 = np.asarray(basis_state(device, 0, i).to_jax()).flatten()
        n1 = np.asarray(basis_state(device, 1, i).to_jax()).flatten()
        overlap_n0 = np.abs(evecs.conj().T @ n0)**2
        overlap_n1 = np.abs(evecs.conj().T @ n1)**2
        idx_n0 = np.argmax(overlap_n0)
        idx_n1 = np.argmax(overlap_n1)
        P_n = np.outer(evecs[:, idx_n0], evecs[:, idx_n1].conj())
        P_comp = P_comp + P_n

    P_comp = dq.asqarray(P_comp,
                     dims=(params.fluxonium_dim,
                           params.resonator_dim))

    return P_comp
    
    
    

In [ ]:
def dressed_fluxonium(params, pulse):

    ops = operators(params, pulse)

    h_resonator = params.omega_r * ops["n_photon"]
    h_fluxonium = ops["h_fluxonium"]

    h_interaction_rwa = (
        params.g * (-1j)
        * (
            ops["a"] @ ops["n_plus"]
            - ops["a"].dag() @ ops["n_minus"]
        )
    )

    H = h_resonator + h_fluxonium + h_interaction_rwa

    H = np.asarray(H.to_jax())

    evals, evecs = np.linalg.eigh(H)

    dressed_indices = set()

    for n in range(params.resonator_dim):

        # bare |0,n>
        g_n = np.asarray(
            basis_state(params, 0, n).to_jax()
        ).flatten()

        overlaps = np.abs(evecs.conj().T @ g_n)**2
        dressed_indices.add(np.argmax(overlaps))

        # bare |1,n>
        e_n = np.asarray(
            basis_state(params, 1, n).to_jax()
        ).flatten()

        overlaps = np.abs(evecs.conj().T @ e_n)**2
        dressed_indices.add(np.argmax(overlaps))

    P_comp = np.zeros_like(H, dtype=complex)

    for idx in dressed_indices:

        psi = evecs[:, idx]

        P_comp += np.outer(
            psi,
            psi.conj()
        )

    return dq.asqarray(P_comp,dims=(params.fluxonium_dim,params.resonator_dim))

In [ ]:
for i in range(1, device.resonator_dim):
    n0 = np.asarray(basis_state(device, i, 0).to_jax()).flatten()
    n1 = np.asarray(basis_state(device, i, 1).to_jax()).flatten()
    overlap_n0 = np.abs(evecs.conj().T @ n0)**2
    overlap_n1 = np.abs(evecs.conj().T @ n1)**2
    idx_n0 = np.argmax(overlap_n0)
    idx_n1 = np.argmax(overlap_n1)
    P_n = np.outer(evecs[: idx_n0], evecs[: idx_n1].conj())
    P_comp = P_comp + P_n
   
    print(i)

NameError: name 'evecs' is not defined

In [ ]:
P_comp = dressed_fluxonium(device, pulse)

/mnt/d/GPU_fluxonium/hamiltonian.py:40: UserWarning: A sparse qarray has been converted to dense layout due to element-wise addition with a dense qarray.
  "h_static": h_resonator + h_fluxonium + params.g * drive_op @ n,
/tmp/ipykernel_931193/601190943.py:16: UserWarning: A sparse qarray has been converted to dense layout due to element-wise addition with a dense qarray.
  H = h_resonator + h_fluxonium + h_interaction_rwa


In [ ]:
print(dq.trace(P_comp))
print(dq.norm(P_comp @ P_comp - P_comp))

(130+4.358085e-10j)
-3.0715325e-06


In [ ]:
from hamiltonian import computational_projector
computational_projector(device)

QArray: shape=(650, 650), dims=(10, 65), dtype=complex64, layout=dense
[[1.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 ...
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]]

In [ ]:
def dressed_fluxonium_computational_projector(params:DeviceParameters, pulse:PulseParameters, fluxonium_state: int): 

    ops = operators(params, pulse)

    #omega_r = Omega_r(params, pulse, fluxonium_state)

    h_resonator = params.omega_r * ops["n_photon"]
    h_fluxonium = ops["h_fluxonium"]

    h_interaction_rwa = (params.g * (-1j)* (ops["a"] @ ops["n_plus"]- ops["a"].dag() @ ops["n_minus"]))

    H = h_resonator + h_fluxonium + h_interaction_rwa

    H = np.asarray(H.to_jax())

    evals, evecs = np.linalg.eigh(H)

    idx0_list = []
    idx1_list = []

    for n in range(params.resonator_dim):

        g_n = np.asarray(basis_state(params, 0, n).to_jax()).flatten()
        overlaps = np.abs(evecs.conj().T @g_n)**2
        idx0 = np.argmax(overlaps)
        idx0_list.append(idx0)

        e_n = np.asarray(basis_state(params, 1, n).to_jax()).flatten()
        overlaps = np.abs(evecs.conj().T @ e_n)**2
        idx1 = np.argmax(overlaps)
        idx1_list.append(idx1)

    print("g_branch", idx0_list)
    print("e_branch", idx1_list)
    print("unique g:", len(set(idx0_list)))
    print("unique e:" , len(set(idx1_list)))
    print("total unique:", len(set(idx0_list + idx1_list)))

    # dressed_indices = set()

    # for n in range(params.resonator_dim):

    #     for i in range(params.omega_levels.shape[0]):

    #         # bare |0,n>
    #         g_n = np.asarray(basis_state(params, i, n).to_jax()).flatten()

    #         overlaps = np.abs(evecs.conj().T @ g_n)**2
    #         dressed_indices.add(np.argmax(overlaps))

    #         # bare |1,n>
    #         e_n = np.asarray(basis_state(params, i+1 , n).to_jax()).flatten()

    #         overlaps = np.abs(evecs.conj().T @ e_n)**2
    #         dressed_indices.add(np.argmax(overlaps))

    # P_comp = np.zeros_like(H, dtype=complex)

    # for idx in dressed_indices:

    #     psi = evecs[:, idx]

    #     P_comp += np.outer(psi,psi.conj())

    # return dq.asqarray(P_comp,dims=(params.fluxonium_dim,params.resonator_dim))

In [ ]:
for n in range(10):
    bare = basis_state(device, 1, n)
    overlaps = np.abs(evecs.conj().T@ bare)**2
    idx = np.argmax(overlaps)
    print(n, idx, overlaps[idx])

In [ ]:
dressed_fluxonium_computational_projector(device, pulse, 1)

/tmp/ipykernel_931193/1411974913.py:12: UserWarning: A sparse qarray has been converted to dense layout due to element-wise addition with a dense qarray.
  H = h_resonator + h_fluxonium + h_interaction_rwa


g_branch [np.int64(0), np.int64(5), np.int64(13), np.int64(25), np.int64(41), np.int64(61), np.int64(81), np.int64(101), np.int64(121), np.int64(141), np.int64(161), np.int64(181), np.int64(201), np.int64(221), np.int64(241), np.int64(261), np.int64(281), np.int64(301), np.int64(321), np.int64(341), np.int64(361), np.int64(381), np.int64(401), np.int64(421), np.int64(441), np.int64(461), np.int64(481), np.int64(501), np.int64(521), np.int64(541), np.int64(561), np.int64(581), np.int64(601), np.int64(621), np.int64(641), np.int64(661), np.int64(681), np.int64(701), np.int64(721), np.int64(741), np.int64(761), np.int64(781), np.int64(801), np.int64(821), np.int64(841), np.int64(861), np.int64(881), np.int64(901), np.int64(921), np.int64(941), np.int64(961), np.int64(981), np.int64(1001), np.int64(1021), np.int64(1041), np.int64(1061), np.int64(1081), np.int64(1101), np.int64(1121), np.int64(1141), np.int64(1161), np.int64(1181), np.int64(1201), np.int64(1221), np.int64(1241)]
e_branch [n

In [ ]:
idx0_list = []
idx1_list = []

for n in range(params.resonator_dim):

    g_n = np.asarray(basis_state(params, 0, n).to_jax().flatten)
    overlaps = np.abs(evecs.conj().T @g_n)**2
    idx0 = np.argmax(overlaps)
    idx0_list.append(idx0)

    e_n = np.asarray(basis_state(params, 1, n).to_jax()).flatten()
    overlaps = np.abs(evecs.conj().T @ e_n)**2
    idx1 = np.argmax(overlaps)
    idx1_list.append(idx1)

print("g_branch", idx0_list)
print("e_branch", idx1_list)
print("unique g:", len(set(idx0_list)))
print("unique e:" , len(set(idx1_list)))
print("total unique:", len(set(idx0_list + idx1_list)))

#TESTING HAMILTONIAN HERMITICITY#

In [ ]:
def testing_hamiltonian_hermiticity(params: DeviceParameters, pulse: PulseParameters, fluxonium_state: int, t_max: float = 350.0, n_points: int = 500):
    h_full = readout_hamiltonian(params, pulse, fluxonium_state)
    t_grid = np.sort(np.concatenate([np.linspace(0.0, t_max, n_points), np.array([99.999, 100.0, 100.001, 299.999, 300.0, 300.001])]))


In [ ]:
x = np.array([[0, 1, 2], [3, 4, 5], [6, 7, 8]])
print(x)
s = np.sum(x[2:, :], axis=0)
s = x[0] + x[1]
print(s)

[[0 1 2]
 [3 4 5]
 [6 7 8]]
[3 5 7]


In [24]:
import numpy as np
import scqubits as scq


def get_fluxonium_charge_matrix(
    EJ: float, EC: float, EL: float,
    flux: float = 0.5,
    cutoff: int = 110,
    truncated_dim: int = 20,
    angular: bool = True,
) -> tuple[np.ndarray, np.ndarray, "scq.Fluxonium"]:
    """
    Diagonalize a fluxonium qubit with scqubits and return eigenfrequencies
    and the charge-operator matrix elements in the qubit eigenbasis, in the
    format expected by params.omega_levels / params.n_matrix.

    Parameters
    ----------
    EJ, EC, EL : float
        Josephson, charging, inductive energies, all in GHz (scqubits' native
        convention -- these are frequencies f = E/h, NOT angular frequencies).
    flux : float
        External flux in units of Phi_0. flux=0.5 is the usual fluxonium
        "sweet spot" (half flux quantum), where charge-parity symmetry is
        restored and n_matrix becomes near-tridiagonal. Away from 0.5,
        n_matrix is generically dense across most (i,j) pairs.
    cutoff : int
        Internal charge/phase basis size scqubits uses for diagonalization.
        Must be large enough for numerical convergence -- check by increasing
        it and confirming the low-lying eigenvalues/matrix elements stop changing.
        This is INDEPENDENT of truncated_dim.
    truncated_dim : int
        Number of low-lying eigenstates to keep -- this becomes your
        params.fluxonium_dim. Convergence in this number is the qubit-level
        truncation check discussed for branch-crossing resolution.
    angular : bool
        If True (default), returned omega_levels are angular frequencies
        (omega = 2*pi*f), matching the convention used throughout this
        codebase (e.g. "omega_r/2pi = 9.37 GHz" style parameters, and time
        in ns inside mesolve). If False, returned in scqubits' native GHz
        (i.e. plain frequency, not angular) -- only use this if your
        DeviceParameters expects that convention instead.

    Returns
    -------
    omega_levels : np.ndarray, shape (truncated_dim,)
        Eigenfrequencies relative to the ground state (omega_levels[0] = 0).
    n_matrix : np.ndarray, shape (truncated_dim, truncated_dim), complex
        Charge operator <i|n_hat|j> in the truncated eigenbasis. Hermitian
        by construction; verified below.
    fluxonium : scq.Fluxonium
        The underlying scqubits object, kept in case you also want
        phi_operator(), potential(), or plotting utilities.
    """
    fluxonium = scq.Fluxonium(
        EJ=EJ, EC=EC, EL=EL, flux=flux,
        cutoff=cutoff, truncated_dim=truncated_dim,
    )

    evals, evecs = fluxonium.eigensys(evals_count=truncated_dim)
    omega_levels = evals - evals[0]

    n_matrix = fluxonium.n_operator(energy_esys=(evals, evecs))

    herm_err = np.max(np.abs(n_matrix - n_matrix.conj().T))
    if herm_err > 1e-8:
        print(f"WARNING: n_matrix Hermiticity error = {herm_err:.2e} -- check cutoff/truncated_dim")

    if angular:
        omega_levels = 2 * np.pi * omega_levels

    return omega_levels, n_matrix, fluxonium


def sweep_fluxonium_charge_matrix(
    EJ_list: list[float], EC_list: list[float], EL_list: list[float],
    flux: float = 0.5, cutoff: int = 110, truncated_dim: int = 20, angular: bool = True,
) -> dict[tuple[float, float, float], dict]:
    """
    Sweep over combinations of (EJ, EC, EL) and return omega_levels/n_matrix
    for each. Useful for scanning device designs before committing to a
    specific set of DeviceParameters.
    """
    results = {}
    for EJ in EJ_list:
        for EC in EC_list:
            for EL in EL_list:
                key = (EJ, EC, EL)
                omega_levels, n_matrix, flx = get_fluxonium_charge_matrix(
                    EJ, EC, EL, flux=flux, cutoff=cutoff,
                    truncated_dim=truncated_dim, angular=angular,
                )
                results[key] = {
                    "omega_levels": omega_levels,
                    "n_matrix": n_matrix,
                    "omega_01": omega_levels[1] - omega_levels[0],
                    "fluxonium": flx,
                }
    return results


def plot_charge_matrix_density(n_matrix: np.ndarray, title: str = ""):
    """
    Quick visual check of |n_ij| -- diagnostic for how many transitions are
    non-negligible (relevant to how many branches can hybridize in the
    readout-branch analysis).
    """
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(np.abs(n_matrix), cmap="viridis")
    ax.set_xlabel("j"); ax.set_ylabel("i")
    ax.set_title(title or r"$|n_{ij}|$")
    plt.colorbar(im, ax=ax)
    return fig


if __name__ == "__main__":
    # single device, at the sweet spot
    omega_levels, n_matrix, flx = get_fluxonium_charge_matrix(
        EJ=4.0, EC=1.0, EL=0.75, flux=0.5, cutoff=110, truncated_dim=20,
    )
    print("omega_01 / 2pi (GHz):", (omega_levels[1] - omega_levels[0]) / (2 * np.pi))

    # sweep example
    results = sweep_fluxonium_charge_matrix(
        EJ_list=[3.5, 4.0, 4.5],
        EC_list=[1.0],
        EL_list=[0.5, 0.75, 1.0],
        flux=0.5, truncated_dim=20,
    )
    for key, r in results.items():
        print(key, "omega_01/2pi (GHz):", r["omega_01"] / (2 * np.pi))

omega_01 / 2pi (GHz): 0.3885477795342029
(3.5, 1.0, 0.5) omega_01/2pi (GHz): 0.31880979723733255
(3.5, 1.0, 0.75) omega_01/2pi (GHz): 0.5157249925599325
(3.5, 1.0, 1.0) omega_01/2pi (GHz): 0.7542968057363133
(4.0, 1.0, 0.5) omega_01/2pi (GHz): 0.23721924373498204
(4.0, 1.0, 0.75) omega_01/2pi (GHz): 0.3885477795342029
(4.0, 1.0, 1.0) omega_01/2pi (GHz): 0.5818489963974649
(4.5, 1.0, 0.5) omega_01/2pi (GHz): 0.17779987270919062
(4.5, 1.0, 0.75) omega_01/2pi (GHz): 0.29265305143051457
(4.5, 1.0, 1.0) omega_01/2pi (GHz): 0.44539423872079986


In [26]:
Ej = 25.132741228718345
#Ec = 0.73032 #/2/pi
Ec = 6.283185307179586 #/2/pi
#El = 0.55097 #/2/pi
El = 4.71238898038469#/2/pi

In [27]:
omega_levels, n_matrix, fluxonium = get_fluxonium_charge_matrix(Ej, Ec, El, 0.5, 200, 26, False)

In [46]:
omega_levels

array([  0.        ,   2.4413177 ,  24.5184862 ,  38.39124286,
        57.69781167,  76.83685633,  96.0083157 , 113.94903159,
       129.66829555, 142.6730112 , 154.27648639, 166.48543106,
       179.83288878, 193.90793579, 208.51635141, 223.62017352,
       239.15995767, 255.06588386, 271.27615196, 287.7306077 ,
       304.36667362, 321.11879921, 337.91724924, 354.68660057,
       371.34473294, 387.80282556])

In [47]:
n_matrix

array([[0.-5.86107085e-32j, 0.-1.14024146e-01j, 0.+1.66357640e-15j,
        0.+4.21002491e-01j, 0.+4.83814377e-16j, 0.+6.86766089e-03j,
        0.-2.47310106e-16j, 0.-2.08849604e-02j, 0.+9.43825607e-16j,
        0.+6.99153291e-03j, 0.+8.45733723e-16j, 0.-2.23622634e-03j,
        0.-7.02317849e-17j, 0.-8.73845425e-05j, 0.-3.54827226e-16j,
        0.-3.91480447e-04j, 0.-3.91606815e-17j, 0.+2.86843596e-04j,
        0.+1.00728066e-17j, 0.-1.27967990e-04j, 0.-5.82613997e-17j,
        0.+3.91426257e-05j, 0.-1.63278513e-17j, 0.-5.61533811e-06j,
        0.+5.03557716e-17j, 0.-2.78150480e-06j],
       [0.+1.14024146e-01j, 0.+5.80482987e-32j, 0.+5.64517383e-01j,
        0.-1.11967578e-15j, 0.-2.12794663e-01j, 0.-1.57911734e-16j,
        0.+4.13752187e-02j, 0.+6.86190640e-16j, 0.-1.59499258e-03j,
        0.+3.77277277e-16j, 0.+4.24104311e-03j, 0.-1.36135009e-15j,
        0.+3.09527701e-03j, 0.+2.88126653e-16j, 0.+1.34272079e-03j,
        0.-1.51865644e-16j, 0.-3.52280168e-04j, 0.+9.91908396e-16j,

In [49]:
print(device.n_matrix.dtype)   # should be complex128, not float64

complex128


In [50]:
import json
import numpy as np

def save_complex_matrix_json(matrix: np.ndarray, filepath: str):
    data = {
        "real": matrix.real.tolist(),
        "imag": matrix.imag.tolist(),
        "shape": list(matrix.shape),
        "dtype": str(matrix.dtype),
    }
    with open(filepath, "w") as f:
        json.dump(data, f)

def load_complex_matrix_json(filepath: str) -> np.ndarray:
    with open(filepath, "r") as f:
        data = json.load(f)
    real = np.array(data["real"])
    imag = np.array(data["imag"])
    return real + 1j * imag

In [51]:
def chi_multilevel(omega_levels, n_matrix, g, omega_r):
    chi = 0.0
    for i_f, target in [(0, +1), (1, -1)]:
        for j in range(len(omega_levels)):
            if j == i_f:
                continue
            omega_ij = omega_levels[j] - omega_levels[i_f]
            g_ij = g * n_matrix[i_f, j]
            chi += target * (abs(g_ij)**2 * omega_ij) / (omega_ij**2 - omega_r**2)
    return chi

# chi scales as g^2 for fixed n_matrix/omega_levels/omega_r -- solve by direct rescaling
g_probe = 1.0
chi_at_probe = chi_multilevel(omega_levels, n_matrix, g_probe, device.omega_r)
chi_target = 2 * np.pi * 2.5e-3   # 2.5 MHz, in the same angular-GHz units as omega_r

g_new = g_probe * np.sqrt(chi_target / chi_at_probe)
print(f"g/2pi needed = {g_new/(2*np.pi)*1e3:.2f} MHz")

g/2pi needed = 283.06 MHz


In [52]:
import json
import numpy as np

def update_device_json_with_complex_n_matrix(filepath: str, n_matrix: np.ndarray, output_path: str = None):
    with open(filepath, "r") as f:
        data = json.load(f)

    #data["device"]["omega_levels"] = omega_levels.tolist()
    data["device"]["n_matrix_real"] = n_matrix.real.tolist()
    data["device"]["n_matrix_imag"] = n_matrix.imag.tolist()
    data["device"].pop("n_matrix", None)          # old real-only array
    data["device"].pop("n_matrix_minus", None)    # dead code, built from the wrong matrix
    data["device"].pop("n_matrix_plus", None)     # dead code, built from the wrong matrix

    with open(output_path or filepath, "w") as f:
        json.dump(data, f, indent=2)

In [53]:
def load_device_params_from_json(filepath: str) -> dict:
    with open(filepath, "r") as f:
        data = json.load(f)
    d = data["device"]
    n_matrix = np.array(d["n_matrix_real"]) + 1j * np.array(d["n_matrix_imag"])
    return {
        "omega_levels": np.asarray(d["omega_levels"]),
        "n_matrix": n_matrix,
        "omega_r": d["omega_r"],
        "g": d["g"],
        "kappa": d["kappa"],
        "resonator_dim": d["resonator_dim"],
    }

In [ ]:
update_device_json_with_complex_n_matrix("/mnt/d/GPU_fluxonium/device_parameter.json", n_matrix=n_matrix)

: 

In [ ]:
load_device_params_from_json("/mnt/d/GPU_fluxonium/device_parameter.json")

{'omega_levels': array([  0.        ,   2.44125645,  24.51829029,  38.39085932,
         57.69726277,  76.83614993,  96.00748727, 113.94813516,
        129.66739968, 142.67212914, 154.27552557, 166.48430725,
        179.83159958, 193.90649291, 208.51475336, 223.61841799,
        239.15804494, 255.06381514, 271.2739293 , 287.72823444,
        304.36415457, 321.11614088]),
 'n_matrix': array([[0.-3.33857408e-32j, 0.-1.14024146e-01j, 0.+1.66282314e-15j,
         0.+4.21002491e-01j, 0.+3.91270488e-16j, 0.+6.86766089e-03j,
         0.-2.49317395e-16j, 0.-2.08849604e-02j, 0.+9.39196407e-16j,
         0.+6.99153291e-03j, 0.+9.15946387e-16j, 0.-2.23622634e-03j,
         0.-5.61972233e-17j, 0.-8.73845425e-05j, 0.-3.94200318e-16j,
         0.-3.91480447e-04j, 0.-7.86807605e-17j, 0.+2.86843596e-04j,
         0.+4.93162229e-17j, 0.-1.27967990e-04j, 0.-5.55258716e-17j,
         0.+3.91426257e-05j],
        [0.+1.14024146e-01j, 0.-1.50534668e-31j, 0.+5.64517383e-01j,
         0.-1.05944964e-15j, 0.-

In [ ]:
params, pulse, sim = load_json("device_parameter.json")
print(params.n_matrix.dtype)   # should be complex128
print(np.allclose(params.n_matrix, params.n_matrix.conj().T))  # Hermiticity check

complex128
True


: 